# Interactive Clean

The CASA interactive clean application was developed to replace the [CASA6 interactive clean functionality](https://casa.nrao.edu/UserMan/UserMansu291.html) both in CASA6 and in future CASA versions. It was initially included in CASA6 with version 6.7.2. Version 6.7.3 adds **basic** Jupyter Notebook support. The primary difference between usage in Jupyter and usage directly from a Python command line interface (CLI) is that the `iclean.notebook` function is used within a notebook instead of the `iclean` function that is used from the Python CLI. The `iclean.notebook( )` will open the interactive clean GUI within a notebook cell. The CLI `iclean` function can also be used in notebooks, but it will open the GUI in a browser tab **outside** of the notebook.

## Limitations

There are limitations with the version of interactive clean included in CASA 6.7.3:

* the Jupyter Kernel must be running on the **same host** as the browser where the notebook is displayed
* reloading a notebook that has been saved with an interactive clean GUI displayed in cell output currently does not work correctly in the reloaded notebook
* there may be bugs with running multiple versions of interactive clean within the same notebook

## Usage

The remainder of this notebook describes how interactive clean can be used within a Jupyter notebook. In general, the `iclean.notebook()` function is used to create the `iclean` displayable object. This object can then be displayed in one of three ways:

* executing the displayable's `show` member function
* *evaluation* (i.e. displayed because it is returned by the **last** statement of the cell)
* Bokeh's `show` function

While all of these display methods can be used, mixing the various display methods within the same notebook is **not** supported. This is a a limitation of the development framework on which this application is based. It is not considered a bug. Unfortunately, what this means for this notebook is that all of the cells below cannot be evaluated without reopening the notebook.

Continue reading to see these options in practice.

## Which version of cubevis to use

The current version of `cubevis` to use is [v1.0.20](https://pypi.org/project/cubevis/1.0.20/).

## Setup
This first cell simply retrieves the data that is required.

In [ ]:
import os
import sys
import ssl
import certifi
import urllib
import tarfile
ms_path = 'refim_point_withline.ms'
ms_url = "https://casa.nrao.edu/download/devel/casavis/data/refim_point_withline-ms.tar.gz"
if not os.path.isdir(ms_path):
    try:
        context = ssl.create_default_context(cafile=certifi.where())
        tstream = urllib.request.urlopen(ms_url, context=context, timeout=400)
        tar = tarfile.open(fileobj=tstream, mode="r:gz")
        tar.extractall( **({'filter': 'data'} if sys.version_info >= (3, 12) else {}) )
    except urllib.error.URLError:
        print("Failed to open connection to "+ms_url)

## Import
The first step is to import the `iclean` function:

In [ ]:
from cubevis import iclean

## Cleanup
`iclean` operates on state contained within files in the current directory, so it is always a good idea to make sure that you are starting with a clean slate. This command below removes the state files that accumulate from running this example, **but make sure that it and subsequent cells do not remove files you want to keep**.

In [ ]:
!rm -rf test.*

## Create the Application Object
This cell creates the interactive clean application object. This object is used to display the GUI and to retreive the `iclean` return dictionary from the GUI once the user has completed cleaning.

In [ ]:
ic = iclean.notebook( vis=ms_path, imagename='test',
                      imsize=512, niter=50,
                      cycleniter=10,
                      cell='12.0arcsec',
                      specmode='cube', interpolation='nearest',
                      nchan=5, start='1.0GHz', width='0.2GHz', pblimit=-1e-05,
                      deconvolver='hogbom', threshold='0.001Jy', cyclefactor=3, scales=[0,3,10] )

## Display the GUI
Once the application object has been created, the GUI can be displayed:

In [ ]:
ic.show( )

## Retrieve the result
Once the user has completed cleaning, clicked the stop sign and approved the exiting the application, the result can be retrieved:

In [ ]:
ic.get_result( )

## Alternate ways to display the GUI
While this is the most obvious way to create and use the interactive clean within a Jupyter notebook, there are also some subtle variations.

### Display as a result of cell evaulation
The last evaluated statement in a notebook cell is displayed and displaying the `iclean` application object results in the GUI being displayed.

In [ ]:
!rm -rf test.*
iclean.notebook( vis=ms_path, imagename='test',
                 imsize=512, niter=50,
                 cycleniter=10,
                 cell='12.0arcsec',
                 specmode='cube', interpolation='nearest',
                 nchan=5, start='1.0GHz', width='0.2GHz', pblimit=-1e-05,
                 deconvolver='hogbom', threshold='0.001Jy', cyclefactor=3, scales=[0,3,10] )

### Retrieving the result
Even though we did not retain the application object, the value of the **last** cell evaluation is available as `_`. So using this, we can retrieve the return dictionary:

In [ ]:
_.get_result( )

It is important to remember that `_` changes with **each** executed statement so the `iclean` displayable object will no longer be available from `_` after another statement is executed

### Using Bokeh's show function
One last slight variation is to use [Bokeh's](https://bokeh.org/) `show` function to display the application object:

In [ ]:
!rm -rf test.*
from bokeh.io import output_notebook
from bokeh.plotting import show
output_notebook( )
ic = iclean.notebook( vis=ms_path, imagename='test',
                      imsize=512, niter=50,
                      cycleniter=10,
                      cell='12.0arcsec',
                      specmode='cube', interpolation='nearest',
                      nchan=5, start='1.0GHz', width='0.2GHz', pblimit=-1e-05,
                      deconvolver='hogbom', threshold='0.001Jy', cyclefactor=3, scales=[0,3,10] )
show(ic)

### Regular `iclean` is still usable

While `iclean.notebook` allows for the interactive clean GUI to be displayed in a notebook cell, the Python CLI version of interctive clean should also be usable:

In [ ]:
!rm -rf test.*
result = iclean( vis=ms_path, imagename='test',
                 imsize=512, niter=50,
                 cycleniter=10,
                 cell='12.0arcsec',
                 specmode='cube', interpolation='nearest',
                 nchan=5, start='1.0GHz', width='0.2GHz', pblimit=-1e-05,
                 deconvolver='hogbom', threshold='0.001Jy', cyclefactor=3, scales=[0,3,10] )

In [ ]:
print(result)